In [86]:
import pandas as pd
import os
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf

In [4]:
os.getcwd()

'/Users/jessie/Desktop/berk/summer 26/207/watts-the-cost-DATASCI-207'

In [77]:
df = pd.read_parquet('data/cleaned_data.parquet')

In [78]:
X = df.drop(columns=['rate'])
y = df['rate']

In [79]:
# dropping zip code
X = X.drop(columns=["zip"])

# log transforming as specified in project milestone
est_cols = [
    "est",
    "n<5",
    "n5_9",
    "n10_19",
    "n20_49",
    "n50_99",
    "n100_249",
    "n250_499",
    "n500_999",
    "n1000"
]

X[est_cols] = np.log1p(X[est_cols])

# converting rate type index into categorical
X["rate_type_index"] = X["rate_type_index"].astype(str)

In [80]:
# utility columns are numbers stored as strings. convert

utility_cols = [
    "Utility Res Sales MWh",
    "Utility Com Sales MWh",
    "Utility Ind Sales MWh"
]

X[utility_cols] = X[utility_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

#impute nas w median (can change to 0 too)
X[utility_cols] = X[utility_cols].fillna(
    X[utility_cols].median()
)

## one hot encoding, scaling numerical vars

In [81]:
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [col for col in X.columns if col not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat",
         OneHotEncoder(drop="first",
                       handle_unknown="ignore",
                      sparse_output=False),
         cat_cols),
        ("num",
         StandardScaler(),
         num_cols)
    ]
)

## splitting data

In [82]:
# 1. Partitioning into training, testing, and validation sets. Splitting into temp and test, then train and validation
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=1234)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=1234)

## Preprocessing

In [83]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

## LR model

In [89]:
def build_model(num_features, learning_rate):

  tf.keras.backend.clear_session()
  tf.random.set_seed(0)

  model = tf.keras.Sequential()
  model.add(tf.keras.layers.Dense(
      units=1,                     # output dim
      input_shape=(num_features,),  # input dim
      use_bias=True,               # use a bias (intercept) param
      kernel_initializer=tf.ones_initializer,  # initialize params to 1
      bias_initializer=tf.ones_initializer,    # initialize bias to 1
  ))

  optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)

  model.compile(
      optimizer=optimizer,
      loss='mse',
      metrics=[
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
            tf.keras.metrics.MeanAbsoluteError(name="mae")
        ]
  )
    
  return model

In [92]:
lr = [0.0001, 0.001]
epochs = [5, 10, 20]

for i in epochs:
    for j in lr:
        model_tf=build_model(X_train_processed.shape[1], j)
        print(f"epochs={i}, lr={j}")
        fit_model_tf = model_tf.fit(
            x=X_train_processed,
            y=y_train,
            epochs=i,
            validation_data=(X_val_processed, y_val),
            verbose=1)
        final_loss = fit_model_tf.history['loss'][-1]
        final_val_loss = fit_model_tf.history['val_loss'][-1]
        print(f"epochs={i}, lr={j} | loss={final_loss:.4f}, val_loss={final_val_loss:.4f}\n")      

epochs=5, lr=0.0001
Epoch 1/5
3131/3131 [==============================] - 13s 4ms/step - loss: 51.1228 - rmse: 7.1500 - mae: 4.0028 - val_loss: 6.0569 - val_rmse: 2.4611 - val_mae: 1.7748
Epoch 2/5
3131/3131 [==============================] - 13s 4ms/step - loss: 3.5005 - rmse: 1.8710 - mae: 1.3191 - val_loss: 1.8577 - val_rmse: 1.3630 - val_mae: 1.0011
Epoch 3/5
3131/3131 [==============================] - 12s 4ms/step - loss: 1.3655 - rmse: 1.1686 - mae: 0.8484 - val_loss: 0.9583 - val_rmse: 0.9789 - val_mae: 0.7279
Epoch 4/5
3131/3131 [==============================] - 12s 4ms/step - loss: 0.7886 - rmse: 0.8880 - mae: 0.6572 - val_loss: 0.6407 - val_rmse: 0.8005 - val_mae: 0.5971
Epoch 5/5
3131/3131 [==============================] - 12s 4ms/step - loss: 0.5579 - rmse: 0.7469 - mae: 0.5553 - val_loss: 0.4871 - val_rmse: 0.6979 - val_mae: 0.5185


epochs=5, lr=0.0001 | loss=0.5579, val_loss=0.4871

epochs=5, lr=0.001
Epoch 1/5
  33/3131 [..............................] - ETA: 9s - loss: 262.7241 - rmse: 16.2088 - mae: 11.0612 

2026-07-29 15:59:34.437309: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node SGD/AssignVariableOp.


3131/3131 [==============================] - 13s 4ms/step - loss: 6.0770 - rmse: 2.4652 - mae: 0.9441 - val_loss: 0.2273 - val_rmse: 0.4768 - val_mae: 0.3398
Epoch 2/5
3131/3131 [==============================] - 12s 4ms/step - loss: 0.1528 - rmse: 0.3909 - mae: 0.2638 - val_loss: 0.1082 - val_rmse: 0.3290 - val_mae: 0.2069
Epoch 3/5
3131/3131 [==============================] - 12s 4ms/step - loss: 0.0862 - rmse: 0.2936 - mae: 0.1762 - val_loss: 0.0725 - val_rmse: 0.2693 - val_mae: 0.1497
Epoch 4/5
3131/3131 [==============================] - 12s 4ms/step - loss: 0.0617 - rmse: 0.2484 - mae: 0.1331 - val_loss: 0.0564 - val_rmse: 0.2374 - val_mae: 0.1172
Epoch 5/5
3131/3131 [==============================] - 12s 4ms/step - loss: 0.0495 - rmse: 0.2226 - mae: 0.1075 - val_loss: 0.0476 - val_rmse: 0.2182 - val_mae: 0.0989


epochs=5, lr=0.001 | loss=0.0495, val_loss=0.0476

epochs=10, lr=0.0001
Epoch 1/10
  30/3131 [..............................] - ETA: 11s - loss: 510.4386 - rmse: 22.5929 - mae: 16.6559

...

2026-07-29 16:04:52.370291: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node SGD/AssignVariableOp.


3131/3131 [==============================] - 13s 4ms/step - loss: 51.1228 - rmse: 7.1500 - mae: 4.0028 - val_loss: 6.0569 - val_rmse: 2.4611 - val_mae: 1.7748
Epoch 2/20
3131/3131 [==============================] - 12s 4ms/step - loss: 3.5005 - rmse: 1.8710 - mae: 1.3191 - val_loss: 1.8577 - val_rmse: 1.3630 - val_mae: 1.0011
Epoch 3/20
3131/3131 [==============================] - 13s 4ms/step - loss: 1.3655 - rmse: 1.1686 - mae: 0.8484 - val_loss: 0.9583 - val_rmse: 0.9789 - val_mae: 0.7279
Epoch 4/20
3131/3131 [==============================] - 12s 4ms/step - loss: 0.7886 - rmse: 0.8880 - mae: 0.6572 - val_loss: 0.6407 - val_rmse: 0.8005 - val_mae: 0.5971
Epoch 5/20
3131/3131 [==============================] - 13s 4ms/step - loss: 0.5579 - rmse: 0.7469 - mae: 0.5553 - val_loss: 0.4871 - val_rmse: 0.6979 - val_mae: 0.5185
Epoch 6/20
3131/3131 [==============================] - 13s 4ms/step - loss: 0.4374 - rmse: 0.6614 - mae: 0.4899 - val_loss: 0.3955 - val_rmse: 0.6289 - val_mae: 0.4

epochs=20, lr=0.0001 | loss=0.1099, val_loss=0.1082

epochs=20, lr=0.001
Epoch 1/20
  27/3131 [..............................] - ETA: 12s - loss: 289.7070 - rmse: 17.0208 - mae: 11.8010 

2026-07-29 16:09:05.140053: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] model_pruner failed: INVALID_ARGUMENT: Graph does not contain terminal node SGD/AssignVariableOp.


3131/3131 [==============================] - 14s 4ms/step - loss: 6.0770 - rmse: 2.4652 - mae: 0.9441 - val_loss: 0.2273 - val_rmse: 0.4768 - val_mae: 0.3398
Epoch 2/20
3131/3131 [==============================] - 14s 4ms/step - loss: 0.1528 - rmse: 0.3909 - mae: 0.2638 - val_loss: 0.1082 - val_rmse: 0.3290 - val_mae: 0.2069
Epoch 3/20
3131/3131 [==============================] - 13s 4ms/step - loss: 0.0862 - rmse: 0.2936 - mae: 0.1762 - val_loss: 0.0725 - val_rmse: 0.2693 - val_mae: 0.1497
Epoch 4/20
3131/3131 [==============================] - 12s 4ms/step - loss: 0.0617 - rmse: 0.2484 - mae: 0.1331 - val_loss: 0.0564 - val_rmse: 0.2374 - val_mae: 0.1172
Epoch 5/20
3131/3131 [==============================] - 13s 4ms/step - loss: 0.0495 - rmse: 0.2226 - mae: 0.1075 - val_loss: 0.0476 - val_rmse: 0.2182 - val_mae: 0.0989
Epoch 6/20
3131/3131 [==============================] - 13s 4ms/step - loss: 0.0424 - rmse: 0.2059 - mae: 0.0915 - val_loss: 0.0417 - val_rmse: 0.2043 - val_mae: 0.08